# Sistema da Análise de Sentimento

In [30]:
# importando as biblitotecas
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, DataCollator, DataCollatorWithPadding
from datasets import Dataset
import torch

In [31]:
model_name = 'neuralmind/bert-base-portuguese-cased'
nlp = pipeline('fill-mask', model= model_name)

texto = "O fruto que cai de uma laranjeira é uma [MASK]."

resultados = nlp(texto)

for r in resultados:
    print(f"Sugestão: {r['sequence']} | Score: {r['score']:.4f}")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 996c8787-d26b-4f93-bc70-282d305c6e50)')' thrown while requesting HEAD https://huggingface.co/neuralmind/bert-base-portuguese-cased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of the model checkpoint at neuralmind/bert-base-portuguese-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with ano

Sugestão: O fruto que cai de uma laranjeira é uma flor. | Score: 0.1421
Sugestão: O fruto que cai de uma laranjeira é uma árvore. | Score: 0.1141
Sugestão: O fruto que cai de uma laranjeira é uma delas. | Score: 0.0548
Sugestão: O fruto que cai de uma laranjeira é uma beleza. | Score: 0.0383
Sugestão: O fruto que cai de uma laranjeira é uma rosa. | Score: 0.0344


In [32]:
data = {
    "text":[
        "Eu adorei este produto, ele é maravilhoso!",
        "Isso é horrível, nunca mais compro.",
        "O atendimento foi excelente e rápido.",
        "Estou muito insatisfeito com a qualidade.",
        "Valeu cada centavo, recomendo muito.",
        "Péssima experiência, odiei."
    ],
    "label":[1,0,1,0,1,0]
}

dataset = Dataset.from_dict(data)

In [33]:
# carregar o tokenizer e Modelo
model_checkpoint = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels= 2)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
# Tokenização dos dados
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation= True, padding="max_length", max_length= 128)

tokenized_dataset = dataset.map(tokenize_function, batched= True)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 6/6 [00:00<00:00, 702.90 examples/s]


In [35]:
# Configuração de treinamento
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 20,
    per_device_train_batch_size= 2,
    logging_steps= 1,
    eval_strategy="no",
    report_to= "none"
)

In [36]:
# Inicializando o trainer com Data Collator
data_collator = DataCollatorWithPadding(tokenizer= tokenizer)

trainer = Trainer(
    model= model,
    args= training_args,
    train_dataset= tokenized_dataset,
    data_collator= data_collator
)

In [37]:
# Inicializando o treinamento
print("Inicializando o fine-tuning corrigido...")
trainer.train()
print("Treinamento concluído!")

Inicializando o fine-tuning corrigido...


  2%|▏         | 1/60 [00:00<00:08,  7.29it/s]

{'loss': 0.7014, 'grad_norm': 4.375002384185791, 'learning_rate': 4.9166666666666665e-05, 'epoch': 0.33}


  3%|▎         | 2/60 [00:00<00:08,  6.82it/s]

{'loss': 0.7138, 'grad_norm': 9.083770751953125, 'learning_rate': 4.8333333333333334e-05, 'epoch': 0.67}


...

{'loss': 0.0041, 'grad_norm': 0.09926831722259521, 'learning_rate': 8.333333333333333e-07, 'epoch': 19.67}


100%|██████████| 60/60 [00:08<00:00,  6.71it/s]

{'loss': 0.0048, 'grad_norm': 0.11088650673627853, 'learning_rate': 0.0, 'epoch': 20.0}


100%|██████████| 60/60 [00:10<00:00,  5.48it/s]

{'train_runtime': 10.9577, 'train_samples_per_second': 10.951, 'train_steps_per_second': 5.476, 'train_loss': 0.1432159946183674, 'epoch': 20.0}
Treinamento concluído!


In [ ]:
# Testando
frase_teste = "Gostei, muito bom!"
inputs = tokenizer(frase_teste, return_tensors="pt", truncation=True, padding= True, max_length= 128)

# Move os dados para o mesmo dispositivo (GPU ou CPU) onde o modelo está
inputs = inputs.to(model.device)

with torch.no_grad():
    logits = model(**inputs).logits

predicao = torch.argmax(logits, dim=1).item()
sentimento = "Positivo" if predicao == 1 else "Negativo"

print(f"Frase: {frase_teste}")
print(f"Sentimento previsto: {sentimento}")

Frase: Gostei, muito bom!
Sentimento previsto: Positivo
